[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/en/lab2/lab2-part2.ipynb)

# Lab 1: Neural networks from scratch — Part 2 — PyTorch

In this second part of the lab we are going to use PyTorch to implement and train the same neural network that we developed with NumPy in Part 1.

We will therefore need the `torch` library in addition to the already used `numpy`, `pandas`, and `seaborn`.

In [ ]:
import torch
import numpy as np
import seaborn as sns
import pandas as pd

# We set a random seed so that the results are reproducible across runs
np.random.seed(1234567)

We will load the `titanic` dataset, just as we did in Part 1, but transforming `X` and `y` into `torch` tensors. We will obtain two tensors (`x_data` and `labels`) that we will use later.

In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler

def load_titanic():
    # Load the Titanic dataset from seaborn
    df = sns.load_dataset('titanic')

    # 1) Select relevant variables and clean
    # Columns we will use
    cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
    df = df[cols].copy()

    # Drop rows with missing values
    df = df.dropna(subset=['age', 'embarked', 'fare'])

    # 2) Split labels and features
    y = df['survived'].to_numpy().astype(np.float32)        # labels as float
    X = df.drop(columns=['survived'])

    # 3) One-hot encoding for all categorical variables
    categorical_cols = ['pclass', 'sex', 'embarked', 'alone']
    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)  # drop_first=True avoids multicollinearity

    # 4) Numeric variables
    numeric_cols = ['age', 'sibsp', 'parch', 'fare']
    X_numeric = X_encoded[numeric_cols + [c for c in X_encoded.columns if c not in numeric_cols]]
    scaler = StandardScaler()
    X_numeric[numeric_cols] = scaler.fit_transform(X_numeric[numeric_cols])

    # 5) Convert to numpy arrays
    X_np = X_numeric.to_numpy().astype(np.float32)
    y_np = y.reshape(-1, 1).astype(np.float32)  # reshape to (n_samples, 1)

    return X_np, y_np

x_data, labels = load_titanic()
x_data = torch.from_numpy(x_data)
labels = torch.from_numpy(labels)

## Model declaration

First, we must create in PyTorch the graph of operations that represents our model. To do so:
 1. We create the variables that the framework will optimize, that is, the parameters of the model.
 1. We create the graph of operations that produce the prediction from the input and the variables. In this case we will use functions that relate learnable variables with tensors that will contain data, using framework operations.

In [ ]:
# Auxiliary variables
input_size = x_data.shape[1]
h0_size = 5
h1_size = 3

# CREATING THE VARIABLES
# TODO - Complete the matrix dimensions
W0 = torch.tensor(np.random.randn(h0_size, input_size), dtype=torch.float32, requires_grad=True)
b0 = torch.tensor(np.random.randn(1, h0_size), dtype=torch.float32, requires_grad=True)
W1 = torch.tensor(np.random.randn(h1_size, h0_size), dtype=torch.float32, requires_grad=True)
b1 = torch.tensor(np.random.randn(1, h1_size), dtype=torch.float32, requires_grad=True)
W2 = torch.tensor(np.random.randn(1, h1_size), dtype=torch.float32, requires_grad=True)
b2 = torch.tensor(np.random.randn(1, 1), dtype=torch.float32, requires_grad=True)

# Store all variables in a list for later access
VARIABLES = [W0, b0, W1, b1, W2, b2]


# BUILD THE COMPUTATION GRAPH
def sigmoid_layer(x, W, b):
    # TODO - Complete the layer output using PyTorch ops in the next line
    return 

def predict(x):
    # TODO - Complete the following lines
    h0 = 
    h1 = 
    y = 
    return y

# Check
x_test = np.random.randn(1,input_size)
y_pred = predict(x_test) 
print(y_pred)
np.testing.assert_almost_equal(0.494716, y_pred.detach().numpy(), err_msg='Check your implementation')

## Training the model

The declared model can already be used to make predictions by passing a tensor with data to the `predict` function (as was done in the verification section of the previous cell). However, as we saw in Part 1, this model is not fitted to the input data, so it will produce bad predictions.

We must find a set of values for the parameters ($\mathbf{W}_2$, $b_2$, $\mathbf{W}_1$, $\mathbf{b}_1$, $\mathbf{W}_0$ and $\mathbf{b}_0$) that minimize the cost function. PyTorch helps us optimize this process.

PyTorch allows us to configure the optimization process, so we must tell it:
 1. Which loss function we want. In our case we had chosen the binary cross entropy.
 1. Which optimization method to use. As in Part 1, we will use gradient descent.

For the moment we will create two variables to store both configurations. Being organized in this way, using a different loss function or a different optimization algorithm becomes as simple as changing these variables.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Loss function (Binary Cross Entropy)
loss_fn = nn.BCELoss()  # expects outputs in [0, 1]

# SGD optimizer with learning rate 0.1
# VARIABLES is the list of tensors with requires_grad=True
optimizer = optim.SGD(VARIABLES, lr=0.1)
# optimizer = optim.Adam(VARIABLES, lr=0.1)

### The training loop

The training loop will be analogous to the one used in Part 1. It will consist of running a preset number (`NUM_EPOCHS`) of training steps. At each step we will do the following:
 1. Take the input data and compute the predictions the model makes in its current state
 1. Compute the cost (the average of the losses of each prediction)
 1. Use the cost value to update each variable in the direction of its gradient

We will create a `training_step` function that does this work. PyTorch will take care of computing the gradients and performing the updates of the variables. To update the parameters, we must call the `backward` function on the tensor that contains the value we want to optimize. Once we have called it, we can ask the optimizer to take a `step` in the direction of gradient descent.

In [ ]:
num_samples = x_data.shape[0]

def training_step(x, y):
    # Zero the gradients
    optimizer.zero_grad()
    
    # TODO - Complete the next line so it computes the predictions
    y_pred = 
    
    # Compute the loss with the function chosen above
    loss = loss_fn(y, y_pred)

    # Compute the loss gradient w.r.t. all tensors with requires_grad=True
    loss.backward()
    
    # Updating the variables only requires this call
    optimizer.step()
    
    # Accuracy
    errors = torch.abs(y.reshape(-1,1) - y_pred)
    accuracy = torch.sum(1 - errors)
    
    # Return these two values so we can print them when useful
    return (loss, accuracy)

# TRAINING LOOP
num_epochs = 10000
for epoch in range(num_epochs):    
    loss, accuracy = training_step(x_data, labels)
    
    if epoch % 100 == 99:
        print("Epoch:", epoch, 'Loss:', loss.numpy(), 'Accuracy:', accuracy.numpy()/num_samples)


Using PyTorch has allowed us to abstract from the implementation details and the computation of derivatives to focus on the architecture of our model.